# ConectaONG — Dashboard de Organizações da Sociedade Civil

### Projeto Integrador — Etapa 2

**Tema:** Distribuição das Organizações da Sociedade Civil (OSCs) no Brasil

**Objetivo:** analisar a distribuição estimada de OSCs por estado e região, gerando informações que apoiem a proposta do ConectaONG de conectar organizações, voluntários e doadores.

Este notebook realiza seleção/carregamento da fonte de dados, preparação e organização dos dados, análise exploratória e criação de um dashboard com pelo menos três visualizações.

## 1. Fonte de dados

A base utilizada é o arquivo `organizacoes_sociais_por_estado_brasil.csv`, fornecido para o projeto.

A própria planilha identifica a fonte como **Ipea / Mapa das OSC (Base 2024-2026)**. Essa informação é mantida conforme aparece no arquivo.

### Variáveis disponíveis
- **Região:** região brasileira da UF.
- **UF:** sigla da unidade federativa.
- **Estado:** nome da unidade federativa.
- **Estimativa de OSCs Ativas:** quantidade estimada de OSCs ativas.
- **Fonte dos Dados:** identificação da fonte informada na base.

> **Observação:** a base possui uma linha por UF. Portanto, para analisar a quantidade de OSCs, devemos utilizar a coluna numérica **Estimativa de OSCs Ativas**, e não contar as linhas por estado.

In [2]:
import pandas as pd
import plotly.express as px
from IPython.display import display, HTML
from google.colab import files

print('Bibliotecas carregadas com sucesso!')

Bibliotecas carregadas com sucesso!


## 2. Carregamento da base

A célula abaixo permite selecionar o CSV diretamente no Google Colab. Isso evita depender de um caminho específico do computador ou do Google Drive.

In [3]:
uploaded = files.upload()

if not uploaded:
    raise ValueError('Nenhum arquivo foi selecionado.')

csv_file_path = next(iter(uploaded))
print(f'Arquivo selecionado: {csv_file_path}')

# A base fornecida utiliza ponto e vírgula como separador.
df = pd.read_csv(csv_file_path, sep=';', encoding='utf-8')

print(f'Base carregada: {df.shape[0]} linhas e {df.shape[1]} colunas.')
display(df.head())

Saving organizacoes_sociais_por_estado_brasil.csv to organizacoes_sociais_por_estado_brasil.csv
Arquivo selecionado: organizacoes_sociais_por_estado_brasil.csv
Base carregada: 27 linhas e 5 colunas.


,Região,UF,Estado,Estimativa de OSCs Ativas,Fonte dos Dados
0,Centro-Oeste,DF,Distrito Federal,18500,Ipea / Mapa das OSC (Base 2024-2026)
1,Centro-Oeste,GO,Goiás,31400,Ipea / Mapa das OSC (Base 2024-2026)
2,Centro-Oeste,MS,Mato Grosso do Sul,12600,Ipea / Mapa das OSC (Base 2024-2026)
3,Centro-Oeste,MT,Mato Grosso,15200,Ipea / Mapa das OSC (Base 2024-2026)
4,Nordeste,AL,Alagoas,10500,Ipea / Mapa das OSC (Base 2024-2026)


## 3. Conhecendo os dados

Antes das análises, verificamos a estrutura da base, os tipos das colunas, valores ausentes e possíveis registros duplicados.

In [4]:
print('Dimensões da base:', df.shape)
print('\nTipos de dados:')
print(df.dtypes)

print('\nValores ausentes por coluna:')
display(df.isnull().sum().to_frame('Valores ausentes'))

print('Registros duplicados:', df.duplicated().sum())

Dimensões da base: (27, 5)

Tipos de dados:
Região                       object
UF                           object
Estado                       object
Estimativa de OSCs Ativas     int64
Fonte dos Dados              object
dtype: object

Valores ausentes por coluna:


,Valores ausentes
Região,0
UF,0
Estado,0
Estimativa de OSCs Ativas,0
Fonte dos Dados,0


Registros duplicados: 0


## 4. Preparação e organização dos dados

Nesta etapa:
- removemos espaços extras dos nomes das colunas;
- padronizamos os campos de texto;
- garantimos que a quantidade estimada de OSCs seja numérica;
- removemos duplicidades, se existirem;
- verificamos se existem valores ausentes nas colunas essenciais.

Esses procedimentos deixam a base pronta para análise.

In [5]:
df.columns = df.columns.str.strip()

for col in ['Região', 'UF', 'Estado', 'Fonte dos Dados']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

col_oscs = 'Estimativa de OSCs Ativas'
df[col_oscs] = pd.to_numeric(df[col_oscs], errors='coerce')

antes = len(df)
df = df.drop_duplicates().copy()
depois = len(df)

print(f'Registros antes da remoção de duplicidades: {antes}')
print(f'Registros depois da remoção de duplicidades: {depois}')

print('\nValores ausentes após o tratamento:')
display(df.isnull().sum().to_frame('Valores ausentes'))

print('\nBase preparada:')
display(df.head())

Registros antes da remoção de duplicidades: 27
Registros depois da remoção de duplicidades: 27

Valores ausentes após o tratamento:


,Valores ausentes
Região,0
UF,0
Estado,0
Estimativa de OSCs Ativas,0
Fonte dos Dados,0



Base preparada:


,Região,UF,Estado,Estimativa de OSCs Ativas,Fonte dos Dados
0,Centro-Oeste,DF,Distrito Federal,18500,Ipea / Mapa das OSC (Base 2024-2026)
1,Centro-Oeste,GO,Goiás,31400,Ipea / Mapa das OSC (Base 2024-2026)
2,Centro-Oeste,MS,Mato Grosso do Sul,12600,Ipea / Mapa das OSC (Base 2024-2026)
3,Centro-Oeste,MT,Mato Grosso,15200,Ipea / Mapa das OSC (Base 2024-2026)
4,Nordeste,AL,Alagoas,10500,Ipea / Mapa das OSC (Base 2024-2026)


## 5. Indicadores principais

Os indicadores abaixo resumem os principais números disponíveis na base.

In [6]:
total_oscs = int(df[col_oscs].sum())
estado_maior = df.loc[df[col_oscs].idxmax(), 'Estado']
valor_maior = int(df[col_oscs].max())
regiao_maior = df.groupby('Região')[col_oscs].sum().idxmax()
valor_regiao_maior = int(df.groupby('Região')[col_oscs].sum().max())

print(f'Total estimado de OSCs: {total_oscs:,}'.replace(',', '.'))
print(f'Estado com maior estimativa: {estado_maior} ({valor_maior:,})'.replace(',', '.'))
print(f'Região com maior estimativa: {regiao_maior} ({valor_regiao_maior:,})'.replace(',', '.'))

display(HTML(f'''<div style="display:flex;gap:12px;flex-wrap:wrap;">
<div style="padding:18px;border:1px solid #ddd;border-radius:10px;min-width:190px;"><b>Total estimado de OSCs</b><br><span style="font-size:24px">{total_oscs:,}</span></div>
<div style="padding:18px;border:1px solid #ddd;border-radius:10px;min-width:190px;"><b>Estado com maior estimativa</b><br><span style="font-size:24px">{estado_maior}</span></div>
<div style="padding:18px;border:1px solid #ddd;border-radius:10px;min-width:190px;"><b>Região com maior estimativa</b><br><span style="font-size:24px">{regiao_maior}</span></div>
</div>'''))

Total estimado de OSCs: 978.800
Estado com maior estimativa: São Paulo (232.000)
Região com maior estimativa: Sudeste (437.700)


## 6. Visualização 1 — OSCs estimadas por estado

Esta visualização compara a estimativa de OSCs ativas entre as unidades federativas.

**Pergunta respondida:** quais estados apresentam as maiores estimativas de OSCs ativas?

In [7]:
df_estado = df.sort_values(col_oscs, ascending=True)

fig_estado = px.bar(
    df_estado,
    x=col_oscs,
    y='Estado',
    orientation='h',
    text=col_oscs,
    title='Estimativa de OSCs Ativas por Estado',
    labels={col_oscs: 'Estimativa de OSCs Ativas', 'Estado': 'Estado'},
    hover_data=['UF', 'Região']
)

fig_estado.update_traces(texttemplate='%{text:,}', textposition='outside')
fig_estado.update_layout(template='plotly_white', height=800, margin=dict(l=20, r=80, t=70, b=20))
fig_estado.show()

## 7. Visualização 2 — OSCs estimadas por região

Os valores estaduais são agregados por região para identificar onde existe maior concentração estimada de OSCs.

**Pergunta respondida:** qual região brasileira concentra a maior estimativa de OSCs?

In [8]:
df_regiao = (
    df.groupby('Região', as_index=False)[col_oscs]
      .sum()
      .sort_values(col_oscs, ascending=False)
)

fig_regiao = px.bar(
    df_regiao,
    x='Região',
    y=col_oscs,
    text=col_oscs,
    title='Estimativa de OSCs Ativas por Região',
    labels={'Região': 'Região', col_oscs: 'Estimativa de OSCs Ativas'}
)

fig_regiao.update_traces(texttemplate='%{text:,}', textposition='outside')
fig_regiao.update_layout(template='plotly_white', height=500)
fig_regiao.show()

## 8. Visualização 3 — Participação das regiões

O gráfico de rosca mostra a participação de cada região no total estimado de OSCs da base.

**Pergunta respondida:** como o total estimado de OSCs está distribuído proporcionalmente entre as regiões?

In [9]:
fig_donut = px.pie(
    df_regiao,
    names='Região',
    values=col_oscs,
    hole=0.45,
    title='Participação das Regiões no Total Estimado de OSCs'
)

fig_donut.update_traces(textposition='inside', textinfo='percent+label')
fig_donut.update_layout(template='plotly_white', height=550)
fig_donut.show()

## 9. Tabela-resumo por região

A tabela abaixo apresenta os valores utilizados na análise regional.

In [10]:
df_regiao_display = df_regiao.copy()
df_regiao_display['Participação (%)'] = (df_regiao_display[col_oscs] / total_oscs * 100).round(2)
display(df_regiao_display)

,Região,Estimativa de OSCs Ativas,Participação (%)
3,Sudeste,437700,44.72
1,Nordeste,245700,25.10
4,Sul,160900,16.44
0,Centro-Oeste,77700,7.94
2,Norte,56800,5.80


## 10. Análise dos resultados

Com base exclusivamente nos dados disponíveis na base utilizada, os resultados calculados acima permitem identificar o estado e a região com maior concentração estimada de OSCs.

### Relação com o ConectaONG

Os resultados reforçam a importância da localização como informação para uma plataforma de conexão entre organizações, voluntários e doadores. Conhecer a distribuição das OSCs ajuda a compreender onde existe maior concentração de organizações e pode apoiar funcionalidades de busca e filtros por localização.

A análise contribui principalmente para os objetivos da Etapa 1 relacionados à divulgação de ONGs e à utilização de filtros por localização.

> **Limitação:** esta base contém apenas informações sobre quantidade estimada de OSCs por UF e região. Ela não possui dados sobre número de voluntários, doadores, campanhas, causas ou valores de doações. Portanto, essas dimensões não devem ser apresentadas como resultados deste dashboard.

In [11]:
print('ANÁLISE AUTOMÁTICA DOS RESULTADOS')
print('-' * 45)
print(f'O estado com maior estimativa é {estado_maior}, com {valor_maior:,} OSCs.'.replace(',', '.'))
print(f'A região com maior estimativa é {regiao_maior}, com {valor_regiao_maior:,} OSCs.'.replace(',', '.'))
print(f'O total estimado na base é de {total_oscs:,} OSCs.'.replace(',', '.'))

participacao_maior_regiao = valor_regiao_maior / total_oscs * 100
print(f'A {regiao_maior} representa aproximadamente {participacao_maior_regiao:.2f}% do total estimado.')

ANÁLISE AUTOMÁTICA DOS RESULTADOS
---------------------------------------------
O estado com maior estimativa é São Paulo. com 232.000 OSCs.
A região com maior estimativa é Sudeste. com 437.700 OSCs.
O total estimado na base é de 978.800 OSCs.
A Sudeste representa aproximadamente 44.72% do total estimado.


## 11. Conclusão

O dashboard permite visualizar de forma objetiva a distribuição estimada das Organizações da Sociedade Civil no Brasil. A preparação dos dados, a agregação por região e as três visualizações facilitam a identificação de estados e regiões com maior concentração de OSCs.

Para o ConectaONG, essas informações podem servir como apoio à organização de uma plataforma com busca e filtros por localização, aproximando usuários das organizações presentes em diferentes regiões do país.

### Próximos passos possíveis

Em uma versão futura do projeto, a base poderia ser ampliada com informações sobre **causa de atuação, campanhas, voluntários, doadores, localização das organizações e arrecadações**, permitindo análises mais próximas de todas as funcionalidades propostas para a plataforma.

## 12. Exportação do dashboard

A célula abaixo gera um arquivo HTML com as três visualizações e indicadores principais. O arquivo pode ser aberto em um navegador e utilizado para demonstrar o funcionamento do dashboard.

In [12]:
dashboard_filename = 'conectaong_dashboard.html'

html_estado = fig_estado.to_html(full_html=False, include_plotlyjs='cdn')
html_regiao = fig_regiao.to_html(full_html=False, include_plotlyjs=False)
html_donut = fig_donut.to_html(full_html=False, include_plotlyjs=False)

dashboard_html = f'''<!DOCTYPE html>
<html lang="pt-BR">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>ConectaONG — Dashboard</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 0; background: #f5f7fa; color: #263238; }}
        .container {{ max-width: 1250px; margin: auto; padding: 24px; }}
        .header {{ background: white; padding: 24px; border-radius: 14px; margin-bottom: 20px; }}
        .header h1 {{ margin: 0 0 8px 0; }}
        .kpis {{ display: flex; gap: 16px; flex-wrap: wrap; margin-bottom: 20px; }}
        .kpi {{ background: white; padding: 20px; border-radius: 14px; flex: 1; min-width: 220px; }}
        .kpi .value {{ font-size: 28px; font-weight: bold; margin-top: 8px; }}
        .card {{ background: white; padding: 16px; border-radius: 14px; margin-bottom: 20px; }}
        .source {{ font-size: 13px; color: #607d8b; margin-top: 20px; }}
    </style>
</head>
<body>
  <div class="container">
    <div class="header">
      <h1>ConectaONG — Dashboard</h1>
      <p>Distribuição estimada de Organizações da Sociedade Civil no Brasil</p>
    </div>
    <div class="kpis">
      <div class="kpi"><b>Total estimado de OSCs</b><div class="value">{total_oscs:,}</div></div>
      <div class="kpi"><b>Estado com maior estimativa</b><div class="value">{estado_maior}</div></div>
      <div class="kpi"><b>Região com maior estimativa</b><div class="value">{regiao_maior}</div></div>
    </div>
    <div class="card">{html_estado}</div>
    <div class="card">{html_regiao}</div>
    <div class="card">{html_donut}</div>
    <div class="source"><b>Fonte informada na base:</b> Ipea / Mapa das OSC (Base 2024-2026).</div>
  </div>
</body>
</html>'''

with open(dashboard_filename, 'w', encoding='utf-8') as f:
    f.write(dashboard_html)

print(f'Dashboard criado com sucesso: {dashboard_filename}')
files.download(dashboard_filename)

Dashboard criado com sucesso: conectaong_dashboard.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>